# Fix Videos: Lag Detection & CSV Synchronization


In [ ]:
import pandas as pd
import os
import glob
import re
import tarfile
import urllib.request
import subprocess
import numpy as np
from scipy.io import wavfile
from scipy.signal import correlate
from pathlib import Path
from collections import defaultdict

# Input: raw OpenFace2 CSVs
OPENFACE2_BASE = "..."

# Original videos
VIDEOS_BASE = "./videos"

# Output: lag-corrected CSVs
RESULT_FOLDER = "..."

LOG_FILE = os.path.join(RESULT_FOLDER, "logs.txt")

AUDIO_SAMPLE_RATE = 44100
EXPECTED_FPS = 50

FFMPEG_DIR = Path.home() / "ffmpeg_bin"
FFMPEG_BINARY = FFMPEG_DIR / "ffmpeg"
FFMPEG_URL = "https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz"

os.makedirs(RESULT_FOLDER, exist_ok=True)

def find_csv(directory):
    """Find the single CSV file in the given directory (searches recursively)."""
    csv_files = glob.glob(os.path.join(directory, "**", "*.csv"), recursive=True)
    if len(csv_files) == 1:
        return csv_files[0]
    elif len(csv_files) == 0:
        raise FileNotFoundError(f"No CSV file found in {directory}")
    else:
        raise ValueError(f"Multiple CSV files found in {directory}: {csv_files}")


def ensure_ffmpeg():
    """Ensure a static ffmpeg binary exists locally; download if missing."""
    if FFMPEG_BINARY.exists():
        print(f"FFmpeg already available at {FFMPEG_BINARY}")
        return

    print(f"Downloading ffmpeg static build from {FFMPEG_URL}")
    archive_path = FFMPEG_DIR / "ffmpeg-release-amd64-static.tar.xz"
    os.makedirs(FFMPEG_DIR, exist_ok=True)
    urllib.request.urlretrieve(FFMPEG_URL, archive_path)
    print("Download complete")

    print("Extracting...")
    with tarfile.open(archive_path) as tar:
        members = [m for m in tar.getmembers() if "ffmpeg" in m.name and not m.isdir()]
        for member in members:
            member.name = os.path.basename(member.name)
            tar.extract(member, path=FFMPEG_DIR)

    print("Cleaning up archive")
    archive_path.unlink()

    if os.name != 'nt':
        os.chmod(FFMPEG_BINARY, 0o755)

    print(f"FFmpeg ready at {FFMPEG_BINARY}")


def extract_audio(video_path, wav_path):
    """Extract mono audio from a video file as a WAV using the local ffmpeg binary."""
    ensure_ffmpeg()
    command = [
        str(FFMPEG_BINARY),
        "-y",
        "-i", video_path,
        "-vn",
        "-acodec", "pcm_s16le",
        "-ar", str(AUDIO_SAMPLE_RATE),
        "-ac", "1",
        wav_path,
    ]
    subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def load_audio(path):
    """Load a WAV file, convert to float32, center, and normalize to [-1, 1]."""
    rate, data = wavfile.read(path)
    data = data.astype(np.float32)
    if data.ndim > 1:
        data = data.mean(axis=1)
    data -= data.mean()
    if np.max(np.abs(data)) > 0:
        data /= np.max(np.abs(data))
    return rate, data


def find_shift(audio1, audio2, rate):
    """Compute temporal shift via cross-correlation (positive means audio1 leads audio2)."""
    corr = correlate(audio1, audio2, mode="full")
    lag = np.argmax(corr) - (len(audio2) - 1)
    return lag / rate


def find_video_for_csv_dir(csv_dir):
    """Given an OpenFace result directory, find the matching original video file."""
    match = re.search(r'(diti_\d+_\d+)', csv_dir)
    if not match:
        raise ValueError(f"Could not extract diti identifier from {csv_dir}")
    diti_id = match.group(1)

    camera_match = re.search(r'Camera\s*(\d+)', csv_dir)
    if not camera_match:
        raise ValueError(f"Could not extract Camera number from {csv_dir}")
    camera_num = camera_match.group(1)

    video_extensions = ['*.avi', '*.mp4', '*.mov', '*.mkv', '*.flv', '*.wmv']
    video_files = []
    for ext in video_extensions:
        pattern = os.path.join(VIDEOS_BASE, diti_id, f"Camera {camera_num}", ext)
        video_files.extend(glob.glob(pattern))

    if len(video_files) == 0:
        raise FileNotFoundError(f"No video found in {os.path.join(VIDEOS_BASE, diti_id, f'Camera {camera_num}')}")
    elif len(video_files) > 1:
        video_files = [f for f in video_files if '_prev' not in f]
        if len(video_files) != 1:
            print(f"  Warning: multiple videos found: {video_files}")

    return video_files[0]


def detect_fps(csv_path):
    """Frame rate from the timestamp delta of the first two rows, or 0 if unreadable."""
    df_head = pd.read_csv(csv_path, nrows=2)
    if len(df_head) >= 2 and 'timestamp' in df_head.columns:
        dt = df_head['timestamp'].iloc[1] - df_head['timestamp'].iloc[0]
        return round(1.0 / dt) if dt > 0 else 0
    return 0


diti_folders = sorted([
    d for d in os.listdir(OPENFACE2_BASE)
    if os.path.isdir(os.path.join(OPENFACE2_BASE, d)) and d.startswith("diti_")
])

print(f"Configuration:")
print(f"  OpenFace results : {OPENFACE2_BASE}")
print(f"  Videos           : {VIDEOS_BASE}")
print(f"  Output           : {RESULT_FOLDER}")
print(f"  Expected fps     : {EXPECTED_FPS}")
print(f"\nFound {len(diti_folders)} diti folders in {OPENFACE2_BASE}")

files = []

for diti_folder in diti_folders:
    print(f"\nProcessing {diti_folder}...")

    infant_dir = os.path.join(OPENFACE2_BASE, diti_folder, "Camera 1")
    adult_dir  = os.path.join(OPENFACE2_BASE, diti_folder, "Camera 2")

    if not os.path.exists(infant_dir) or not os.path.exists(adult_dir):
        print(f"  Skipping {diti_folder}: missing Camera directories")
        continue

    try:
        infant_video = find_video_for_csv_dir(infant_dir)
        adult_video  = find_video_for_csv_dir(adult_dir)

        print(f"  Infant video: {infant_video}")
        print(f"  Adult video:  {adult_video}")

        infant_wav = f"temp_infant_{diti_folder}.wav"
        adult_wav  = f"temp_adult_{diti_folder}.wav"

        print(f"  Extracting audio...")
        try:
            extract_audio(infant_video, infant_wav)
            extract_audio(adult_video, adult_wav)

            rate1, audio1 = load_audio(infant_wav)
            rate2, audio2 = load_audio(adult_wav)
            if rate1 != rate2:
                raise ValueError(f"Sampling rates do not match: {rate1} vs {rate2}")

            lag = find_shift(audio1, audio2, rate1)
        except Exception as e:
            print(f"   Error extracting audio or calculating lag: {e}")
            lag = 0.0  # Default to zero lag on error

        print(f"  Calculated lag: {lag:.3f} seconds")

        for tmp in [infant_wav, adult_wav]:
            try:
                os.remove(tmp)
            except OSError:
                pass

        files.append((f"{diti_folder}/", lag, f"{infant_dir}/", f"{adult_dir}/"))

    except Exception as e:
        print(f"  Error processing {diti_folder}: {e}")
        continue

print(f"\n\nSuccessfully prepared {len(files)} file pairs:")
for name, lag, _, _ in files:
    print(f"  {name}: lag = {lag:.3f}s")


In [ ]:
meeting_sessions = defaultdict(list)
for entry in files:
    name = entry[0].rstrip("/")
    parts = name.split("_")
    meeting_id = "_".join(parts[:2])
    meeting_sessions[meeting_id].append(entry)

session_fps = {}
for meeting_id, entries in meeting_sessions.items():
    for name, lag_seconds, infant_dir, adult_dir in entries:
        sname = name.rstrip("/")
        for label, cam_dir in [("infant", infant_dir), ("adult", adult_dir)]:
            try:
                csv_path = find_csv(cam_dir)
                fps = detect_fps(csv_path)
            except Exception:
                fps = 0
            session_fps[(sname, label)] = fps

# Drop the meeting unless both cameras are EXPECTED_FPS in every session
meetings_to_exclude = set()
fps_report = []

for meeting_id, entries in meeting_sessions.items():
    for name, _, _, _ in entries:
        sname = name.rstrip("/")
        ifps = session_fps.get((sname, "infant"), 0)
        afps = session_fps.get((sname, "adult"), 0)
        if ifps != EXPECTED_FPS or afps != EXPECTED_FPS:
            fps_report.append((sname, ifps, afps))
            meetings_to_exclude.add(meeting_id)

if fps_report:
    print(f"Not {EXPECTED_FPS} fps on both cameras — excluded:")
    for sname, ifps, afps in fps_report:
        print(f"  {sname}: infant={ifps} fps, adult={afps} fps")

if meetings_to_exclude:
    print(f"\nExcluding meetings: {sorted(meetings_to_exclude)}")

files_filtered = [
    entry for entry in files
    if "_".join(entry[0].rstrip("/").split("_")[:2]) not in meetings_to_exclude
]

print(f"\nSessions after FPS filter: {len(files_filtered)} / {len(files)} "
      f"(excluded {len(files) - len(files_filtered)} from "
      f"{len(meetings_to_exclude)} meetings)")

open(LOG_FILE, "w").close()

def pr(msg, tab='   '):
    """Print to stdout and append to the log file."""
    txt = f'{tab}{msg}'
    print(txt)
    with open(LOG_FILE, "a") as f:
        print(txt, file=f)


def reset_frame_indices(df):
    """Reset the 0-indexed Frame column to a contiguous range after row removal."""
    df['Frame'] = range(len(df))


for (name, lag_seconds, infant_dir, adult_dir) in files_filtered:
    sname      = name.rstrip("/")
    infant_csv = find_csv(infant_dir)
    adult_csv  = find_csv(adult_dir)

    infant = pd.read_csv(infant_csv)
    adult  = pd.read_csv(adult_csv)

    pr(f"Name: {name}\nInfant: {infant_csv}\nAdult: {adult_csv}", tab='')
    pr(f"Infant size: {len(infant)} rows × {len(infant.columns)} cols")
    pr(f"Adult size:  {len(adult)} rows × {len(adult.columns)} cols")

    frames_del = round(lag_seconds * EXPECTED_FPS)
    pr(f"Lag: {lag_seconds:.3f}s -> {frames_del} frames to remove at {EXPECTED_FPS} fps")

    # Trim leading frames from whichever camera started earlier
    if frames_del > 0:
        pr(f"Removing {frames_del} leading rows from infant (Camera 1)")
        infant = infant.iloc[frames_del:].reset_index(drop=True)
    elif frames_del < 0:
        frames_del = -frames_del
        pr(f"Removing {frames_del} leading rows from adult (Camera 2)")
        adult = adult.iloc[frames_del:].reset_index(drop=True)

    pr(f"New infant size: {len(infant)}")
    pr(f"New adult size:  {len(adult)}")

    # Trim trailing rows so both CSVs have the same length
    if len(infant) > len(adult):
        to_del = len(infant) - len(adult)
        pr(f"Trimming {to_del} trailing rows from infant (Camera 1) to equalize lengths")
        infant = infant.iloc[:-to_del].reset_index(drop=True)
    elif len(infant) < len(adult):
        to_del = len(adult) - len(infant)
        pr(f"Trimming {to_del} trailing rows from adult (Camera 2) to equalize lengths")
        adult = adult.iloc[:-to_del].reset_index(drop=True)

    reset_frame_indices(infant)
    reset_frame_indices(adult)

    infant_out_dir = os.path.join(RESULT_FOLDER, name, "Camera 1")
    adult_out_dir  = os.path.join(RESULT_FOLDER, name, "Camera 2")
    os.makedirs(infant_out_dir, exist_ok=True)
    os.makedirs(adult_out_dir,  exist_ok=True)

    infant.to_csv(os.path.join(infant_out_dir, "infant.csv"), index=False)
    adult.to_csv(os.path.join(adult_out_dir,   "adult.csv"),  index=False)
    pr(f"Saved -> {infant_out_dir}/infant.csv  ({len(infant)} rows)")
    pr(f"Saved -> {adult_out_dir}/adult.csv    ({len(adult)} rows)")
    pr("", tab='')
